In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import numpy as np
import pandas as pd

def _add_date_features(df):
    df = df.copy()
    df['year']        = df['Date'].dt.year
    df['month']       = df['Date'].dt.month
    df['week_of_year'] = df['Date'].dt.isocalendar().week.astype(int)
    df['quarter']     = df['Date'].dt.quarter
    df['day_of_week'] = df['Date'].dt.dayofweek
    df['week_sin'] = np.sin(2 * np.pi * df['week_of_year'] / 52.178)
    df['week_cos'] = np.cos(2 * np.pi * df['week_of_year'] / 52.178)
    
    return df

In [ ]:
LAG_STEPS = [1, 2, 4, 52]
def _add_lag_features(df, lag_steps=LAG_STEPS):
    df = df.copy()
    grp = df.groupby(['Store', 'Dept'])['Weekly_Sales']

    for lag in lag_steps:
        df[f'sales_lag_{lag}'] = grp.shift(lag)
        

    return df

In [ ]:
ROLLING_WINDOWS = [4, 8, 12]

def _add_rolling_features(df, windows=ROLLING_WINDOWS):
    df = df.copy()
    grp = df.groupby(['Store', 'Dept'])['Weekly_Sales']

    shifted = grp.shift(1)

    for w in windows:
        df[f'rolling_mean_{w}'] = grp.transform(lambda x: x.shift(1).rolling(window=w, min_periods=1).mean())
        df[f'rolling_std_{w}']  = grp.transform(lambda x: x.shift(1).rolling(window=w, min_periods=1).std())

    return df

In [ ]:
def _add_holiday_features(df):
    df = df.copy()
    df['holiday_weight'] = df['IsHoliday'].map({1: 5, 0: 1})

    
    for name in ['Super Bowl', 'Thanksgiving', 'Christmas', 'Labor Day']:
        col = 'is_' + name.lower().replace(' ', '_')
        df[col] = (df['Holiday_Name'] == name).astype(int)

    return df

In [ ]:
def _add_store_dept_features(df):
    df = df.copy()

    type_map = {'A': 0, 'B': 1, 'C': 2}
    df['store_type_enc'] = df['Type'].map(type_map)

    train_mask = df['IsTestSet'] == 0

    store_avg = (
        df[train_mask].groupby('Store')['Weekly_Sales']
        .mean().rename('store_avg_sales')
    )
    dept_avg = (
        df[train_mask].groupby('Dept')['Weekly_Sales']
        .mean().rename('dept_avg_sales')
    )
    store_dept_avg = (
        df[train_mask].groupby(['Store','Dept'])['Weekly_Sales']
        .mean().rename('store_dept_avg_sales')
    )

    df = df.join(store_avg, on='Store')
    df = df.join(dept_avg, on='Dept')
    df = df.join(store_dept_avg, on=['Store','Dept'])

    return df

In [ ]:
def generate_features(
    df: pd.DataFrame,
    lag_steps: list = LAG_STEPS,
    rolling_windows: list = ROLLING_WINDOWS,
    save_plots: bool = True,
    plots_dir: str = "plots/features",
) -> pd.DataFrame:
    print("Building features")
    df = _add_date_features(df)
    print("Date features done")
    df = _add_lag_features(df, lag_steps)
    print(f"Lag features: {lag_steps} done")
    df = _add_rolling_features(df, rolling_windows)
    print(f"Rolling features: {rolling_windows} done")
    df = _add_holiday_features(df)
    print("Holiday features done")
    df = _add_store_dept_features(df)
    print("Store/Dept encoding done")

    if save_plots:
        os.makedirs(plots_dir, exist_ok=True)
        _plot_lag_correlation(df, plots_dir)
        _plot_feature_coverage(df, plots_dir)
        print(f"Charts saved to {plots_dir}/")

    print(f"Final shape: {df.shape}")
    return df